In [ ]:
%load_ext autoreload
%autoreload 2

# Funnel Trouver sa convention collective

Analyse du parcours de l'outils trouver sa CC pour voir où l'on perd les utilisateurs

**Attention : cette analyse s'appuie sur les données de l'API visites (via la BDD extraite). Il faut donc avoir accès à ces informations pour l'utiliser (non compatible RGPD)**

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

interval_start = '2026-07-01 00:00:00'
interval_stop  = '2026-08-01 00:00:00'   # juillet complet (borne haute exclue)

columns = ['action_id',
    'idvisit',
    'actions',
    'operatingsystemname',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'experiments']

# Sous-requête : idvisit ayant au moins une page vue sur les pages ciblées
subquery_idvisits = f"""
    SELECT DISTINCT idvisit
    FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp <  '{interval_stop}'
      AND action_type = 'action'
      AND (
            action_url LIKE '%/convention-collective'
         OR action_url LIKE '%/convention-collective?%'
      )
"""

query_visits = f"""
    SELECT {", ".join(columns)}
    FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp <  '{interval_stop}'
      AND idvisit IN ({subquery_idvisits})
    ORDER BY idvisit, action_timestamp ASC;
"""

visits_data = await matomo.run_query(query_visits)
visits_df = pd.DataFrame(visits_data, columns=columns)

On nettoie l'URL, puis on ne garde que les idvisit qui contiennent au moins une des trois pages de recherche CC.

In [ ]:
from urllib.parse import urlparse

# 1. Clean : on retire tout ce qui suit le "?"
visits_df['action_url'] = visits_df['action_url'].str.split('?').str[0]

# 2. Extraction du path (robuste au domaine) + normalisation du "/" final
def get_path(url):
    if pd.isna(url):
        return None
    return urlparse(url).path.rstrip('/')

visits_df['action_path'] = visits_df['action_url'].apply(get_path)

# 3. Pages de recherche CC ciblées
cc_search_paths = {
    '/outils/convention-collective',
    '/outils/convention-collective/entreprise',
    '/outils/convention-collective/convention',
}

# 4. idvisit ayant au moins une de ces pages
visits_to_keep = visits_df.loc[
    visits_df['action_path'].isin(cc_search_paths), 'idvisit'
].unique()

# 5. On filtre le DataFrame sur ces visites
visits_df = visits_df[visits_df['idvisit'].isin(visits_to_keep)].copy()

On tronque chaque visite pour ne garder qu'à partir de la première page /outils/convention-collective ou /convention-collective, et on isole à part les visites qui se retrouvent vides.

In [ ]:
# Pages "point d'entrée" à partir desquelles on garde le parcours
start_paths = {'/outils/convention-collective', '/convention-collective'}

kept_parts = []
emptied_visits = []   # idvisit sans page de départ trouvée

for idvisit, grp in visits_df.groupby('idvisit', sort=False):
    is_start = grp['action_path'].isin(start_paths).to_numpy()
    if is_start.any():
        first_idx = is_start.argmax()      # position de la 1re page de départ
        kept_parts.append(grp.iloc[first_idx:])
    else:
        emptied_visits.append(idvisit)

# DataFrame des visites conservées (tronquées)
trimmed_df = (
    pd.concat(kept_parts, ignore_index=True)
    if kept_parts else visits_df.iloc[0:0].copy()
)

# DataFrame à part : visites supprimées (aucune page de départ) — lignes complètes
removed_df = visits_df[visits_df['idvisit'].isin(emptied_visits)].copy()

# Stats de sortie
total_visits   = visits_df['idvisit'].nunique()
removed_count  = len(emptied_visits)
kept_count     = trimmed_df['idvisit'].nunique()

print(f"Visites traitées           : {total_visits}")
print(f"Visites conservées         : {kept_count}")
print(f"Visites supprimées (vides) : {removed_count}")

# On remplace visits_df par la version tronquée
visits_df = trimmed_df

On classifie les étapes du parcours

In [ ]:
# --- Classification des pages -------------------------------------------------
def classify(path):
    if path is None:
        return None
    if path == '/outils/convention-collective':                 return 'base_outils'
    if path == '/convention-collective':                        return 'base_cc'
    if path == '/outils/convention-collective/entreprise':      return 'entreprise'
    if path == '/outils/convention-collective/convention':      return 'convention'
    if path.startswith('/outils/convention-collective/entreprise/'): return 'entreprise_siret'
    if path.startswith('/convention-collective/'):              return 'convention_result'
    return 'autre'

# On raisonne sur les pages vues uniquement (navigation)
pages = visits_df[visits_df['action_type'] == 'action'].copy()
pages['cat'] = pages['action_path'].apply(classify)

On parcours les visites pour identifier les comportements des utilisateurs afin de construire le funnel (passage à l'étape suivante, retour en arrière etc.)

In [ ]:
base = {'base_outils', 'base_cc'}
search_pages = {'base_outils', 'base_cc', 'convention', 'entreprise', 'entreprise_siret'}

def split_parcours(cats):
    """Un parcours démarre à chaque page de base. On inclut la base suivante comme
       sentinelle pour pouvoir détecter les retours immédiats (has_next)."""
    base_idx = [i for i, c in enumerate(cats) if c in base]
    segments = []
    for j, start in enumerate(base_idx):
        end = base_idx[j+1] + 1 if j + 1 < len(base_idx) else len(cats)
        segments.append(cats[start:end])
    return segments

def has_subseq(cats, steps):
    it = 0
    for c in cats:
        if c in steps[it]:
            it += 1
            if it == len(steps):
                return True
    return False

def has_next(cats, a_set, b_set):
    for i in range(len(cats) - 1):
        if cats[i] in a_set and cats[i + 1] in b_set:
            return True
    return False

records = []
for idvisit, grp in pages.groupby('idvisit', sort=False):
    cats = grp['cat'].tolist()
    cset = set(cats)
    records.append({
        'idvisit': idvisit,
        'Parcours_Entreprise':            'entreprise' in cset,
        'Parcours_CC':                    'convention' in cset,
        'Parcours_Entreprise_CC':         'entreprise_siret' in cset,
        'Parcours_CC_completed':          has_subseq(cats, [{'convention'}, {'convention_result'}]),
        'Parcours_Entreprise_completed':  has_subseq(cats, [{'entreprise_siret'}, {'convention_result'}]),
        'Parcours_Entreprise_Back':       has_next(cats, {'entreprise'}, base),
        'Parcours_CC_Back':               has_next(cats, {'convention'}, base),
        'Parcours_Entreprise_CC_back':    has_next(cats, {'entreprise_siret'}, {'entreprise'}),
        'Parcours_CC_completed_Back':         has_subseq(cats, [{'convention'}, {'convention_result'}, search_pages]),
        'Parcours_Entreprise_completed_Back': has_subseq(cats, [{'entreprise_siret'}, {'convention_result'}, search_pages]),
    })

funnel_df = pd.DataFrame(records)   # 1 ligne = 1 visite (booléens)

# d = nb de branches distinctes lancées dans la visite (0, 1 ou 2)
d = funnel_df['Parcours_CC'].astype(int) + funnel_df['Parcours_Entreprise'].astype(int)

funnel = pd.Series({
    # +1 par branche lancée ; les visites "base-only" (aucune recherche) comptent 1 => perte
    'Start': int(d.clip(lower=1).sum()),
    **{col: int(funnel_df[col].sum()) for col in funnel_df.columns if col != 'idvisit'}
})

On ajout le device pour avoir le funnel desktop vs mobile

In [ ]:
# OS par visite (constant sur la visite -> on prend la 1re valeur)
os_by_visit = pages.groupby('idvisit', sort=False)['operatingsystemname'].first()
funnel_df['os'] = funnel_df['idvisit'].map(os_by_visit)

mobile_os  = {'Android', 'iOS'}
desktop_os = {'Windows', 'Mac', 'GNU/Linux', 'Chrome OS', 'Ubuntu'}

def to_device(os):
    if os in mobile_os:
        return 'mobile'
    if os in desktop_os:
        return 'desktop'
    return 'autre'                              # valeurs non mappées -> à traiter

funnel_df['device'] = funnel_df['os'].apply(to_device)
print(funnel_df['device'].value_counts(dropna=False))

Création du graph de funnel global

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

f = funnel

nodes = {
    'Start':                         ('Start',              0, 0.0),
    'Parcours_CC':                   ('CC',                 2,  1.3),
    'Parcours_CC_completed':         ('CC completed',       4,  1.3),
    'Parcours_Entreprise':           ('Entreprise',         2, -1.3),
    'Parcours_Entreprise_CC':        ('Entreprise CC',      4, -1.3),
    'Parcours_Entreprise_completed': ('Entreprise compl.',  6, -1.3),
}

forward = [
    ('Start', 'Parcours_CC'),
    ('Parcours_CC', 'Parcours_CC_completed'),
    ('Start', 'Parcours_Entreprise'),
    ('Parcours_Entreprise', 'Parcours_Entreprise_CC'),
    ('Parcours_Entreprise_CC', 'Parcours_Entreprise_completed'),
]
start_edges = {('Start', 'Parcours_CC'), ('Start', 'Parcours_Entreprise')}

# (src, dst, compteur_back, dénominateur du taux)
backward = [
    ('Start', 'Parcours_CC',                       'Parcours_CC_Back',            'Parcours_CC'),
    ('Start', 'Parcours_Entreprise',               'Parcours_Entreprise_Back',    'Parcours_Entreprise'),
    ('Parcours_Entreprise', 'Parcours_Entreprise_CC', 'Parcours_Entreprise_CC_back', 'Parcours_Entreprise_CC'),
    ('Parcours_CC', 'Parcours_CC_completed',       'Parcours_CC_completed_Back',  'Parcours_CC_completed'),
    ('Parcours_Entreprise_CC', 'Parcours_Entreprise_completed',
                                 'Parcours_Entreprise_completed_Back', 'Parcours_Entreprise_completed'),
]

BOX_W, BOX_H = 1.0, 0.55
FWD, BACK = '#2563eb', '#dc2626'
centers = {k: (x, y) for k, (_, x, y) in nodes.items()}

def anchors(src, dst):
    (x0, y0), (x1, y1) = centers[src], centers[dst]
    return (x0 + BOX_W/2, y0), (x1 - BOX_W/2, y1)

def at(p0, p1, t):
    return (p0[0] + t*(p1[0]-p0[0]), p0[1] + t*(p1[1]-p0[1]))

fig, ax = plt.subplots(figsize=(13, 5.5))

for key, (label, x, y) in nodes.items():
    ax.add_patch(FancyBboxPatch((x-BOX_W/2, y-BOX_H/2), BOX_W, BOX_H,
                 boxstyle="round,pad=0.02,rounding_size=0.08",
                 lw=1.2, edgecolor='#334155', facecolor='#f1f5f9', zorder=3))
    ax.text(x, y, f"{label}\n{int(f[key])}", ha='center', va='center',
            fontsize=10, fontweight='bold', zorder=4)

# --- flèches aller ---
for src, dst in forward:
    p0, p1 = anchors(src, dst)
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=16,
                 color=FWD, lw=1.8, zorder=2, shrinkA=0, shrinkB=0))
    if (src, dst) in start_edges:
        continue  # perte affichée en combiné (voir ci-dessous)
    loss = (f[src]-f[dst])/f[src]*100 if f[src] else 0
    lx, ly = at(p0, p1, 0.58)
    ax.text(lx, ly, f"-{loss:.1f}%", ha='center', va='center', color=FWD,
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.85))

# --- perte combinée du Start (CC + Entreprise) ---
retained   = f['Parcours_CC'] + f['Parcours_Entreprise']
start_loss = (f['Start'] - retained)/f['Start']*100 if f['Start'] else 0
ax.text(1.0, 0.0, f"-{start_loss:.1f}%", ha='center', va='center', color=FWD,
        fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', fc='white', ec=FWD, alpha=0.95))

# --- flèches retour ---
for src, dst, bkey, dkey in backward:
    p0, p1 = anchors(src, dst)
    rad = 0.32
    ax.add_patch(FancyArrowPatch(p1, p0, arrowstyle='-|>', mutation_scale=13,
                 color=BACK, lw=1.4, ls='--', zorder=1, shrinkA=0, shrinkB=0,
                 connectionstyle=f"arc3,rad={rad}"))
    rate = f[bkey]/f[dkey]*100 if f[dkey] else 0
    # position sur le sommet de l'arc rouge (offset perpendiculaire à la corde)
    mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
    dx, dy = p0[0]-p1[0], p0[1]-p1[1]      # posA=p1 -> posB=p0
    k = 1.1                                # >0.5 => au-delà du sommet de la courbe
    lx, ly = mx + k*rad*dy, my - k*rad*dx
    ax.text(lx, ly, f"↩ {rate:.1f}%", ha='center', va='center', color=BACK,
            fontsize=9,
            bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.85))

ax.set_xlim(-1, 7.2); ax.set_ylim(-2.4, 2.4); ax.axis('off')
ax.set_title("Funnel Convention Collective — perte (bleu) / taux de retour (rouge)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Création du funnel pour desktop et mobile

In [ ]:
counter_cols = [c for c in funnel_df.columns if c not in ('idvisit', 'os', 'device')]

def compute_funnel(df):
    d = df['Parcours_CC'].astype(int) + df['Parcours_Entreprise'].astype(int)
    return pd.Series({
        'Start': int(d.clip(lower=1).sum()),
        **{c: int(df[c].sum()) for c in counter_cols}
    })

funnels = pd.DataFrame({
    'mobile':  compute_funnel(funnel_df[funnel_df['device'] == 'mobile']),
    'desktop': compute_funnel(funnel_df[funnel_df['device'] == 'desktop']),
    'total':   compute_funnel(funnel_df),
})

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# --- définitions constantes (communes aux deux graphes) ---
nodes = {
    'Start':                         ('Start',              0, 0.0),
    'Parcours_CC':                   ('CC',                 2,  1.3),
    'Parcours_CC_completed':         ('CC completed',       4,  1.3),
    'Parcours_Entreprise':           ('Entreprise',         2, -1.3),
    'Parcours_Entreprise_CC':        ('Entreprise CC',      4, -1.3),
    'Parcours_Entreprise_completed': ('Entreprise compl.',  6, -1.3),
}
forward = [
    ('Start', 'Parcours_CC'),
    ('Parcours_CC', 'Parcours_CC_completed'),
    ('Start', 'Parcours_Entreprise'),
    ('Parcours_Entreprise', 'Parcours_Entreprise_CC'),
    ('Parcours_Entreprise_CC', 'Parcours_Entreprise_completed'),
]
start_edges = {('Start', 'Parcours_CC'), ('Start', 'Parcours_Entreprise')}
backward = [
    ('Start', 'Parcours_CC',                       'Parcours_CC_Back',            'Parcours_CC'),
    ('Start', 'Parcours_Entreprise',               'Parcours_Entreprise_Back',    'Parcours_Entreprise'),
    ('Parcours_Entreprise', 'Parcours_Entreprise_CC', 'Parcours_Entreprise_CC_back', 'Parcours_Entreprise_CC'),
    ('Parcours_CC', 'Parcours_CC_completed',       'Parcours_CC_completed_Back',  'Parcours_CC_completed'),
    ('Parcours_Entreprise_CC', 'Parcours_Entreprise_completed',
                                 'Parcours_Entreprise_completed_Back', 'Parcours_Entreprise_completed'),
]
BOX_W, BOX_H = 1.0, 0.55
FWD, BACK = '#2563eb', '#dc2626'
centers = {k: (x, y) for k, (_, x, y) in nodes.items()}

def anchors(src, dst):
    (x0, y0), (x1, y1) = centers[src], centers[dst]
    return (x0 + BOX_W/2, y0), (x1 - BOX_W/2, y1)

def at(p0, p1, t):
    return (p0[0] + t*(p1[0]-p0[0]), p0[1] + t*(p1[1]-p0[1]))

def draw_funnel(ax, f, title):
    for key, (label, x, y) in nodes.items():
        ax.add_patch(FancyBboxPatch((x-BOX_W/2, y-BOX_H/2), BOX_W, BOX_H,
                     boxstyle="round,pad=0.02,rounding_size=0.08",
                     lw=1.2, edgecolor='#334155', facecolor='#f1f5f9', zorder=3))
        ax.text(x, y, f"{label}\n{int(f[key])}", ha='center', va='center',
                fontsize=10, fontweight='bold', zorder=4)

    # flèches aller + % de perte
    for src, dst in forward:
        p0, p1 = anchors(src, dst)
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=16,
                     color=FWD, lw=1.8, zorder=2, shrinkA=0, shrinkB=0))
        if (src, dst) in start_edges:
            continue
        loss = (f[src]-f[dst])/f[src]*100 if f[src] else 0
        lx, ly = at(p0, p1, 0.58)
        ax.text(lx, ly, f"-{loss:.1f}%", ha='center', va='center', color=FWD,
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.85))

    # perte combinée du Start
    retained   = f['Parcours_CC'] + f['Parcours_Entreprise']
    start_loss = (f['Start'] - retained)/f['Start']*100 if f['Start'] else 0
    ax.text(1.0, 0.0, f"-{start_loss:.1f}%", ha='center', va='center', color=FWD,
            fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', fc='white', ec=FWD, alpha=0.95))

    # flèches retour + taux de retour
    for src, dst, bkey, dkey in backward:
        p0, p1 = anchors(src, dst)
        rad = 0.32
        ax.add_patch(FancyArrowPatch(p1, p0, arrowstyle='-|>', mutation_scale=13,
                     color=BACK, lw=1.4, ls='--', zorder=1, shrinkA=0, shrinkB=0,
                     connectionstyle=f"arc3,rad={rad}"))
        rate = f[bkey]/f[dkey]*100 if f[dkey] else 0
        mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
        dx, dy = p0[0]-p1[0], p0[1]-p1[1]
        k = 1.1
        lx, ly = mx + k*rad*dy, my - k*rad*dx
        ax.text(lx, ly, f"↩ {rate:.1f}%", ha='center', va='center', color=BACK,
                fontsize=9, bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.85))

    ax.set_xlim(-1, 7.2); ax.set_ylim(-2.4, 2.4); ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')

# --- deux graphes empilés : mobile puis desktop ---
fig, axes = plt.subplots(2, 1, figsize=(13, 11))
draw_funnel(axes[0], funnels['mobile'],  "Funnel Convention Collective — MOBILE")
draw_funnel(axes[1], funnels['desktop'], "Funnel Convention Collective — DESKTOP")
plt.tight_layout()
plt.show()